In [ ]:
#pip install matplotlib-venn

In [ ]:
import pandas as pd
import glob
import os
import hashlib
import pandas as pd
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib_venn import venn2, venn3
from collections import defaultdict

In [ ]:
def get_md5sum(x):
    return hashlib.md5(x.encode("utf-8")).hexdigest()[:10]

In [ ]:
def map_assembler(cell):
    if "carpedeam" in cell:
        return "CarpeDeam"
    elif "penguin" in cell:
        return "PenguiN"
    elif "megahit" in cell:
        return "MEGAHIT"
    elif "spades" in cell:
        return "metaSPAdes"
    else:
        return cell  # Return the cell as is if none of the conditions are met

In [ ]:
def map_assembler_config(cell):
    if "carpedeam" in cell and "configUnsafe" in cell:
        return "CarpeDeam\n(unsafe mode)"
    elif "carpedeam" in cell and "configSafe" in cell:
        return "CarpeDeam\n(safe mode)"
    elif "penguin" in cell:
        return "PenguiN"
    elif "megahit" in cell:
        return "MEGAHIT"
    elif "spades" in cell:
        return "metaSPAdes"
    else:
        return cell  # Return the cell as is if none of the conditions are met

In [ ]:
def adjust_assemblerconfig(row):
    if row["assembler_clean"] == "CarpeDeam":
        if "carpedeamSafe" in row["assemblerconfig"]:
            return "CarpeDeam\n(safe mode)"
        elif "carpedeamUnsafe" in row["assemblerconfig"]:
            return "CarpeDeam\n(unsafe mode)"
        else:
            return "CarpeDeam"
    else:
        return row["assembler_clean"]

In [ ]:
def find_files(directory, suffix):
    aln_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith(suffix):
                aln_files.append(os.path.join(root, file))
    return aln_files

In [ ]:
def plot_unique_hits(dfs, titles, dataset, number):


    assembler_order=["CarpeDeam\n(safe mode)", "CarpeDeam\n(unsafe mode)", "MEGAHIT", "PenguiN", "metaSPAdes"]
    assembler_colors= ['#a1c9f4', '#b9f2f0', '#ffb482','#8de5a1','#ff9f9b']

    # Ensure the input is a list of DataFrames and titles
    if not isinstance(dfs, list) or not all(isinstance(df, pd.DataFrame) for df in dfs):
        raise ValueError("Input must be a list of pandas DataFrames.")

    # Extract unique hits per assembler
    unique_hits = [df['reference'].nunique() for df in dfs]

    # Create a data frame for plotting
    data = pd.DataFrame({'Assembler': titles, 'Unique Hits': unique_hits})

    # Set the order of assemblers if provided
    if assembler_order:
        data = data.set_index('Assembler').reindex(assembler_order).reset_index()

    # Plotting
    fig, ax = plt.subplots(figsize=(10, 6))

    # Plot horizontal bar chart using seaborn
    sns.barplot(x='Unique Hits', y='Assembler', data=data, palette=assembler_colors, ax=ax)

    # Rotate x-axis labels
    ax.set_xlabel('Number of Unique Hits', fontsize=12)
    ax.set_ylabel('Assembler', fontsize=12)

    # Add values to bars
    for i, v in enumerate(data['Unique Hits']):
        if not pd.isna(v):
            ax.text(v + 0.1, i, str(int(v)), va='center', fontsize=12)
        else:
            ax.text(0.1, i, 'NaN', va='center', fontsize=12)

    # Adjust layout
    plt.tight_layout()
    plt.show()

In [ ]:
def curate_report_df(file):
    """
    Returns a list of dataframes. Each dataframe belongs to a file/assembler. The analyzed files are from mmseq taxonomy:
    (1) Query
    (2) Target
    (3) Seq.Id.
    (4) Alignment Length
    (5) Number of mismatches
    (6) number of gap openings
    (7) Start in Query
    (8) End in Query
    (9) Start in Target
    (10) End in Target
    (11) Eval
    (12) bit score
    """
    
    df_aln = pd.read_csv(file, sep='\t',names=["query", "target", "seq.Id.", "alnLen", "MM", "gaps", "startQuery", "EndQuery", "startTarget", \
                                                "EndTarget", "Eval", "bit score", "queryLen", "targetLen", "queryCov", "targetCov"])
    df_aln["reference"] = df_aln["target"].str.rsplit('_', n=1).str[1]

    df_aln = df_aln[ df_aln["seq.Id."] >= 0.3 ]
    df_aln = df_aln[ df_aln["Eval"] <= 1e-12 ]
    df_aln = df_aln[ df_aln["alnLen"] >= 100 ]
    df_aln = df_aln.sort_values(by=["Eval"], ascending=True)
    df_aln = df_aln.drop_duplicates(subset=["query"], keep='first')
    df_aln = df_aln.drop_duplicates(subset=["reference"], keep='first')

    df_aln.reset_index(inplace=True)
    
    df_aln["file"] = os.path.basename(file)
    df_aln["dataLabel"]=df_aln["file"].str.split(".raw.raw.").str[0]
    df_aln["assembler"] = df_aln["file"].apply(lambda file: map_assembler(file))
    df_aln["assembler_final"] = df_aln["file"].apply(lambda file: map_assembler_config(file))

    return df_aln

In [ ]:
### TOPHIT REPORT ANALYSIS

path="data/empirical/results/assembly-mmseqs"

assemblers = ['megahit.config10', 'carpedeam.configUnsafe', 'carpedeam.configSafe', 'penguin.config10', 'spades.config10'] 

dic = {}

files_aln = find_files(path, ".tsv")

for file in files_aln:
    if any(assembler in file for assembler in assemblers):
        #print(file)
        #if ("TAF" in file or "ECO" in file) and os.stat(file).st_size != 0:
        if os.stat(file).st_size != 0:
            # Extract the sample ID from the file name
            sample = os.path.basename(file)
            sampleId = sample.split(".raw-raw.")[0]

            results = curate_report_df(file)

            if results.empty == False:
            
                # Initialize the nested dictionary for readname if it doesn't exist
                if sampleId not in dic:
                    dic[sampleId] = {}
                # Curate results and assign to dictionary
                #readname = results["dataLabel"][0]  # Assuming this is correct
                assembler_id = results["assembler_final"][0]
                #print(assembler_id)

                # Store the results
                #dic[sampleId][readname][assembler_id] = results
                dic[sampleId][assembler_id] = results


In [ ]:
#results

In [ ]:
def extract_group_prefix(dataset):
    return ''.join([char for char in dataset if char.isalpha()][:3])  # Extract the first three letters


# Function to plot a combined heatmap with datasets as rows and assemblers as columns
def plot_combined_unique_hits_heatmap_rows(dic, perc):
    assembler_order = ["CarpeDeam\n(safe mode)", "CarpeDeam\n(unsafe mode)", "MEGAHIT", "PenguiN", "metaSPAdes"]

    # Group datasets by their prefixes
    grouped_data = defaultdict(lambda: defaultdict(dict))
    for dataset in dic.keys():
        group = extract_group_prefix(dataset)
        for assembler, df in dic[dataset].items():
            #print("assembler", assembler)
            if assembler in assembler_order:
                unique_hits = df['reference'].nunique()
                grouped_data[group][assembler][dataset] = unique_hits
                #print("grouped_data", grouped_data)

    # Combine all groups into one heatmap
    all_datasets = []
    heatmap_data = pd.DataFrame(columns=assembler_order)

    for group, assemblers_data in grouped_data.items():
        # Collect all datasets in this group and sort them
        datasets = sorted({dataset for assembler_data in assemblers_data.values() for dataset in assembler_data.keys()})
        all_datasets.extend(datasets)

        # Add data for this group to the heatmap
        group_data = pd.DataFrame(index=datasets, columns=assembler_order)
        for assembler in assembler_order:
            for dataset in datasets:
                group_data.loc[dataset, assembler] = assemblers_data.get(assembler, {}).get(dataset, np.nan)
        
        # Concatenate to the main heatmap data
        heatmap_data = pd.concat([heatmap_data, group_data])

    # Convert data to numeric for the heatmap
    heatmap_data = heatmap_data.astype(float)

    # Plot the combined heatmap
    plt.figure(figsize=(10, len(all_datasets) // 2 ))  # Adjust height dynamically based on the number of datasets

    ax = sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".0f",
        cmap="viridis",
        cbar_kws={'label': 'Number of Unique Hits'},
        linewidths=0.5
    )
    
    # Adjust the font size of the color bar legend
    cbar = ax.collections[0].colorbar
    cbar.ax.yaxis.label.set_size(14)
    
    # Customize the plot
    plt.title(f"Unique Hits by Datasets and Assemblers", fontsize=14)
    plt.xlabel('Assembler', fontsize=14)
    plt.ylabel('Dataset', fontsize=14)
    plt.tight_layout()

    # Show or save the plot
    plt.show()




In [ ]:
# Example usage
# Assuming dic is the dictionary of dictionaries with datasets and their results
plot_combined_unique_hits_heatmap_rows(dic, 30)